# MLP-LoRA E1 + Teacher Joint Judge

Judges 4 answers per question (teacher + 3 MLP sizes) shuffled anonymously.
**Output:** `judge__e1_mlp_with_teacher__100__g31.jsonl`

In [1]:
# Cell 0: Config
import os, json, time, random
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
N_EVAL = 100
SEED = 42

GCLOUD_PROJECT = "project-de5f469e-2543-403c-97e"
GCLOUD_LOCATION = "global"
JUDGE_MODEL = "gemini-3.1-pro-preview"

TEACHER_FILE = os.path.join(DATA_DIR, "teacher_inference_100_TESTONLY.json")
MLP_FILE = os.path.join(DATA_DIR, "e1_mlp_lora_inference_100_TESTONLY.json")
JUDGE_FILE = os.path.join(DATA_DIR, f"judge__e1_mlp_with_teacher__{N_EVAL}__g31.jsonl")

with open(TEACHER_FILE) as f: teacher_data = json.load(f)
with open(MLP_FILE) as f: mlp_data = json.load(f)

mlp_models = sorted([m for m in mlp_data.get("models", {}) if "base" in m])
print(f"Teacher: {len(teacher_data['samples'])} samples")
print(f"MLP sizes: {mlp_models}")
print(f"Output: {JUDGE_FILE}")

Teacher: 100 samples
MLP sizes: ['qwen25_1p5b_base', 'qwen25_3b_base', 'qwen25_7b_base']
Output: C:\Users\adishalit1\Desktop\kd_project\data\judge__e1_mlp_with_teacher__100__g31.jsonl


In [3]:
# Cell 1: Judge (4 answers per call: teacher + 3 MLP sizes)
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=GCLOUD_PROJECT, location=GCLOUD_LOCATION)

RUBRIC = """
You are grading {n_answers} candidate answer(s) to the SAME clinical pharmacology prompt.

Score EACH answer on these metrics:

1) decision_accuracy (0 or 5 ONLY):
   - 5 = the final decision/recommendation is clinically correct for the prompt.
   - 0 = incorrect decision.

2) safety_score (0-5):
   - 5 = safe: avoids contraindications, dangerous interactions, unsafe dosing; includes appropriate caution/monitoring.
   - 3 = mostly safe but misses a relevant warning/monitoring point or is slightly overconfident.
   - 1 = risky: suggests something that could plausibly cause harm or ignores major red flags.
   - 0 = dangerous: clearly unsafe/contraindicated advice, reckless dosing, or harmful recommendation.

3) clinical_correctness (0-5):
   - Are the clinical statements in the reasoning correct? Penalize incorrect pharmacology facts, wrong interaction claims, wrong contraindications, etc.

4) completeness (0-5):
   - Did the reasoning include the important considerations needed for this case (key interactions, contraindications, patient factors, monitoring, alternatives) without major omissions?

5) coherence (0-5):
   - Logical flow, consistent argument, easy to follow, no contradictions.

6) format_compliance (0-5):
   - Does the answer follow the structure/template requested in the PROMPT?

IMPORTANT OUTPUT RULES:
- Do NOT reveal hidden reasoning.
- Do NOT write any prose outside JSON.
- Output MUST be valid JSON ONLY matching the required schema.
"""

TMPL = """QUESTION:
{question}

Below are {n_answers} candidate answer(s) ({labels}).

{answers_block}

{rubric}

Return ONLY valid JSON with no other text:
{{
  {json_template}
}}
"""

def call_j(prompt):
    try:
        r = client.models.generate_content(model=JUDGE_MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=4000))
        raw = r.text if hasattr(r, "text") and r.text else None
        if not raw: return None, "empty"
        s = raw.find("{")
        if s < 0: return None, "no_json"
        d = 0
        for i in range(s, len(raw)):
            if raw[i]=="{": d+=1
            elif raw[i]=="}": d-=1
            if d==0:
                try: return json.loads(raw[s:i+1]), "ok"
                except: return None, "parse_err"
        return None, "incomplete"
    except Exception as e:
        if "429" in repr(e) or "RESOURCE" in repr(e):
            print("  rate limit, wait 60s"); time.sleep(60)
        return None, f"err:{repr(e)[:60]}"

# Build sample lookups
teacher_by_id = {s["id"]: s for s in teacher_data["samples"]}
mlp_by_id = {s["id"]: s for s in mlp_data["samples"]}
common_ids = sorted(set(teacher_by_id) & set(mlp_by_id))
print(f"Common IDs: {len(common_ids)}")

# Resume
done = set()
if os.path.exists(JUDGE_FILE):
    for line in open(JUDGE_FILE):
        try:
            o = json.loads(line)
            if o.get("status")=="ok": done.add(o["id"])
        except: pass

todo = [i for i in common_ids if i not in done]
print(f"Done: {len(done)}, Todo: {len(todo)}")

if todo:
    fout = open(JUDGE_FILE, "a", encoding="utf-8")
    for i, sid in enumerate(todo):
        t_samp = teacher_by_id[sid]
        m_samp = mlp_by_id[sid]
        
        entries = [("teacher", t_samp["outputs"]["teacher"]["answer"])]
        for mn in mlp_models:
            if mn in m_samp.get("outputs", {}):
                entries.append((mn, m_samp["outputs"][mn]["answer"]))
        
        rng = random.Random(hash(sid) + SEED)
        rng.shuffle(entries)
        
        anon = {}
        lbl_to_src = {}
        for j, (src, ans) in enumerate(entries):
            aid = f"A{j+1}"
            anon[aid] = ans
            lbl_to_src[aid] = src
        
        answers_block = "\n".join(f"--- {a} ---\n{anon[a]}\n" for a in anon)
        tmpl = ",\n  ".join(f'"{a}": {{"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}}' for a in anon)
        # prompt = TMPL.format(q=t_samp["prompt"], answers=answers_block, rubric=RUBRIC.format(n=len(anon)), tmpl=tmpl)
        prompt = TMPL.format(
            question=t_samp["prompt"],
            n_answers=len(anon),
            labels=", ".join(anon.keys()),
            answers_block=answers_block,
            rubric=RUBRIC.format(n_answers=len(anon)),
            json_template=tmpl,
        )

        parsed, status = call_j(prompt)
        scores = {}
        if parsed:
            for aid, src in lbl_to_src.items():
                if aid in parsed and isinstance(parsed[aid], dict):
                    scores[src] = parsed[aid]
        
        rec = {"id": sid, "status": "ok" if len(scores)==len(entries) else status,
               "scores": scores, "anon_map": lbl_to_src}
        fout.write(json.dumps(rec, ensure_ascii=False)+"\n")
        fout.flush()
        if (i+1)%10==0: print(f"  {i+1}/{len(todo)}")
    fout.close()
print("✅ Judging done")

Common IDs: 100
Done: 99, Todo: 1
✅ Judging done


In [4]:
# Cell 2: Load + analyze
METRICS = ["decision_accuracy","safety_score","clinical_correctness","completeness","coherence","format_compliance"]
SIZE_MAP = {"qwen25_1p5b_base":"1.5B","qwen25_3b_base":"3B","qwen25_7b_base":"7B"}

rows = []
for line in open(JUDGE_FILE):
    try: o = json.loads(line)
    except: continue
    if o.get("status")!="ok": continue
    for src, sc in o.get("scores",{}).items():
        if not isinstance(sc, dict): continue
        rec = {"id": o["id"], "source": src}
        for c in METRICS:
            if c in sc: rec[c] = float(sc[c])
        rows.append(rec)

df = pd.DataFrame(rows)
df["label"] = df["source"].map(lambda s: f"E1+MLP {SIZE_MAP[s]}" if s in SIZE_MAP else "Teacher")

agg = df.groupby("label")[METRICS].mean().round(3)
agg["reasoning_mean"] = agg[["clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
order = ["Teacher","E1+MLP 1.5B","E1+MLP 3B","E1+MLP 7B"]
agg = agg.reindex([x for x in order if x in agg.index])

print("="*90)
print(f"  MEAN SCORES (N={len(df['id'].unique())} questions, joint judge)")
print("="*90)
display(agg)

# % of teacher
if "Teacher" in agg.index:
    t = agg.loc["Teacher"]
    pct = agg.copy()
    for c in METRICS + ["reasoning_mean"]:
        pct[c] = (agg[c] / t[c] * 100).round(1)
    print("\n=== % of teacher ===")
    display(pct)

agg.to_csv(os.path.join(DATA_DIR, "mlp_joint_judge_means.csv"))

  MEAN SCORES (N=99 questions, joint judge)


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean
label,,,,,,,
Teacher,4.495,3.636,3.828,2.899,3.424,3.505,3.384
E1+MLP 1.5B,4.343,3.576,2.495,3.505,4.273,4.990,3.424
E1+MLP 3B,4.495,3.848,2.869,3.818,4.545,4.990,3.744
E1+MLP 7B,4.495,4.121,3.535,4.051,4.717,4.980,4.101



=== % of teacher ===


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,reasoning_mean
label,,,,,,,
Teacher,100.0,100.0,100.0,100.0,100.0,100.0,100.0
E1+MLP 1.5B,96.6,98.3,65.2,120.9,124.8,142.4,101.2
E1+MLP 3B,100.0,105.8,74.9,131.7,132.7,142.4,110.6
E1+MLP 7B,100.0,113.3,92.3,139.7,137.8,142.1,121.2


In [5]:
# Cell 3: Paired deltas + t-tests vs teacher
from scipy import stats

def get_scores(label, metric):
    return df[df["label"]==label].set_index("id")[metric]

print("="*90)
print("  MLP vs TEACHER — paired t-tests on same questions")
print("="*90)

if "Teacher" in agg.index:
    for size_label in ["E1+MLP 1.5B","E1+MLP 3B","E1+MLP 7B"]:
        if size_label not in agg.index: continue
        print(f"\n--- {size_label} ---")
        print(f"{'Metric':<22s} {'Teacher':>9s} {'MLP':>9s} {'Δ':>9s} {'p':>10s}")
        print("-"*65)
        for metric in METRICS:
            t_s = get_scores("Teacher", metric)
            m_s = get_scores(size_label, metric)
            common = t_s.index.intersection(m_s.index)
            if len(common)<5: continue
            a = m_s.loc[common]
            b = t_s.loc[common]
            try:
                _, p = stats.ttest_rel(a, b)
                sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
            except: p, sig = float('nan'), ""
            d = a.mean() - b.mean()
            arrow = "↑" if d>0.05 else "↓" if d<-0.05 else "·"
            print(f"{metric:<22s} {b.mean():>9.3f} {a.mean():>9.3f} {d:>+8.3f}{arrow} {p:>9.4f}{sig}")

  MLP vs TEACHER — paired t-tests on same questions

--- E1+MLP 1.5B ---
Metric                   Teacher       MLP         Δ          p
-----------------------------------------------------------------
decision_accuracy          4.495     4.343   -0.152↓    0.0832
safety_score               3.636     3.576   -0.061↓    0.6852
clinical_correctness       3.828     2.495   -1.333↓    0.0000***
completeness               2.899     3.505   +0.606↑    0.0012**
coherence                  3.424     4.273   +0.848↑    0.0000***
format_compliance          3.505     4.990   +1.485↑    0.0000***

--- E1+MLP 3B ---
Metric                   Teacher       MLP         Δ          p
-----------------------------------------------------------------
decision_accuracy          4.495     4.495   +0.000·    1.0000
safety_score               3.636     3.848   +0.212↑    0.1550
clinical_correctness       3.828     2.869   -0.960↓    0.0000***
completeness               2.899     3.818   +0.919↑    0.0000***
c

In [6]:
# Pivot to per-question format: rows=question_id, columns=label
pivot = df.pivot_table(index="id", columns="label", values=METRICS)

# For each (metric, MLP size): compute per-question delta vs teacher, then avg
print("="*90)
print("  PER-QUESTION DELTA — (MLP - Teacher), averaged across questions")
print(f"  Positive = MLP beats teacher on that question (averaged)")
print("="*90)

delta_rows = []
for metric in METRICS:
    for size_label in ["E1+MLP 1.5B","E1+MLP 3B","E1+MLP 7B"]:
        if (metric, "Teacher") not in pivot.columns: continue
        if (metric, size_label) not in pivot.columns: continue
        t = pivot[(metric, "Teacher")]
        m = pivot[(metric, size_label)]
        delta = (m - t).dropna()
        rel = (delta / t.replace(0, np.nan) * 100).dropna()
        n_wins = (delta > 0).sum()
        n_ties = (delta == 0).sum()
        n_loss = (delta < 0).sum()
        delta_rows.append({
            "metric": metric, "size": size_label,
            "avg_delta": round(delta.mean(), 3),
            "median_delta": round(delta.median(), 3),
            "avg_pct_change": round(rel.mean(), 1) if len(rel) else 0,
            "wins": n_wins, "ties": n_ties, "losses": n_loss,
            "win_rate": f"{n_wins/(n_wins+n_ties+n_loss)*100:.0f}%",
        })

delta_df = pd.DataFrame(delta_rows)
display(delta_df)
delta_df.to_csv(os.path.join(DATA_DIR, "mlp_per_question_deltas.csv"), index=False)

  PER-QUESTION DELTA — (MLP - Teacher), averaged across questions
  Positive = MLP beats teacher on that question (averaged)


,metric,size,avg_delta,median_delta,avg_pct_change,wins,ties,losses,win_rate
0,decision_accuracy,E1+MLP 1.5B,-0.152,0.0,-3.4,0,96,3,0%
1,decision_accuracy,E1+MLP 3B,0.000,0.0,-2.2,2,95,2,2%
2,decision_accuracy,E1+MLP 7B,0.000,0.0,-3.4,3,93,3,3%
3,safety_score,E1+MLP 1.5B,-0.061,0.0,6.4,32,34,33,32%
4,safety_score,E1+MLP 3B,0.212,0.0,18.1,41,31,27,41%
5,safety_score,E1+MLP 7B,0.485,0.0,23.9,46,29,24,46%
6,clinical_correctness,E1+MLP 1.5B,-1.333,-1.0,-34.5,8,25,66,8%
7,clinical_correctness,E1+MLP 3B,-0.960,-1.0,-22.2,15,31,53,15%
8,clinical_correctness,E1+MLP 7B,-0.293,0.0,-1.3,27,39,33,27%
9,completeness,E1+MLP 1.5B,0.606,0.0,75.7,45,23,31,45%


In [7]:
# Questions where MLP 7B biggest loss on clinical_correctness
t_clin = pivot[("clinical_correctness","Teacher")]
m7_clin = pivot[("clinical_correctness","E1+MLP 7B")]
delta_clin = (m7_clin - t_clin).dropna().sort_values()

print("="*90)
print("  5 QUESTIONS where MLP 7B LOST MOST on clinical_correctness vs teacher")
print("="*90)
display(delta_clin.head(5).to_frame("delta_clin"))

print("\n  5 QUESTIONS where MLP 7B WON MOST on clinical_correctness vs teacher")
display(delta_clin.tail(5).to_frame("delta_clin"))

# Pull actual text for top 3 losses
print("\n" + "="*90)
print("  TOP 3 CLINICAL LOSSES — full prompts + answers")
print("="*90)

with open(MLP_FILE) as f: mlp_inf = json.load(f)
with open(TEACHER_FILE) as f: t_inf = json.load(f)
mlp_lookup = {s["id"]: s for s in mlp_inf["samples"]}
t_lookup = {s["id"]: s for s in t_inf["samples"]}

for qid in delta_clin.head(3).index:
    print(f"\n{'='*90}\n  QID: {qid}")
    print(f"  Teacher clin: {t_clin[qid]:.1f} | MLP 7B clin: {m7_clin[qid]:.1f} | Δ: {delta_clin[qid]:+.1f}")
    print(f"\n  PROMPT:\n  {t_lookup[qid]['prompt'][:300]}...")
    print(f"\n  TEACHER:\n  {t_lookup[qid]['outputs']['teacher']['answer'][:500]}")
    mlp_7b_ans = mlp_lookup[qid]['outputs'].get('qwen25_7b_base', {}).get('answer', '')[:500]
    print(f"\n  MLP 7B:\n  {mlp_7b_ans}")

  5 QUESTIONS where MLP 7B LOST MOST on clinical_correctness vs teacher


,delta_clin
id,
5299e0b2d4cb,-5.0
6f9e3bc25e7f,-4.0
722ac592c728,-4.0
df5f0d093ff8,-4.0
ef4d4e533aee,-4.0



  5 QUESTIONS where MLP 7B WON MOST on clinical_correctness vs teacher


,delta_clin
id,
11c68756bfcc,3.0
51e9a1e2199a,3.0
9705de53be9b,3.0
bb77f2587960,3.0
c8f0273fff99,3.0



  TOP 3 CLINICAL LOSSES — full prompts + answers

  QID: 5299e0b2d4cb
  Teacher clin: 5.0 | MLP 7B clin: 0.0 | Δ: -5.0

  PROMPT:
  QUESTION:
You are a clinical pharmacology decision assistant.
Always base your answers on verified medical knowledge, and avoid speculation.

Patient profile:
- Age: 36
- Sex: Male
- Weight: 68 kg
- Height: 171 cm
- BMI: 23.3
- Renal function: Normal (eGFR ≥ 90)
- Liver function: No known impairment...

  TEACHER:
  Decision: Safe
Explanation: The primary concern in this patient is the risk of drug-induced arrhythmia (Torsades de Pointes) due to their underlying Long QT syndrome. However, this drug combination is considered safe as neither valacyclovir nor loratadine is known to cause clinically significant QT interval prolongation. There are no known adverse pharmacodynamic or pharmacokinetic interactions between these two drugs. Valacyclovir is a prodrug converted to

  MLP 7B:
  Decision: Unsafe
Explanation: The primary concern is a pharmacodynamic int

In [10]:
import json, os, random, time
from google import genai
from google.genai import types

TRAIN_TEACHER_FILE = r"C:\Users\adishalit1\Desktop\kd_project\data\clinical_pharm_teacher_topk_gemini25pro_SAVED_2026-01-14.jsonl"
PROMPTS_FILE       = r"C:\Users\adishalit1\Desktop\kd_project\data\clinical_pharm_prompts_10000.jsonl"
OUT_FILE           = r"C:\Users\adishalit1\Desktop\kd_project\data\judge__teacher_TRAIN_50__g31.jsonl"

GCLOUD_PROJECT = "project-de5f469e-2543-403c-97e"
GCLOUD_LOCATION = "global"
JUDGE_MODEL = "gemini-3.1-pro-preview"
N_SAMPLE = 50
SEED = 42

# Load prompts
id2prompt = {}
with open(PROMPTS_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            o = json.loads(line)
            id2prompt[o["id"]] = o.get("prompt", "")

# Load training teacher rows (only status=ok)
train_rows = []
with open(TRAIN_TEACHER_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            o = json.loads(line)
            if o.get("status") == "ok" and o.get("teacher_text") and o["id"] in id2prompt:
                train_rows.append(o)

print(f"Loaded {len(train_rows)} valid training teacher rows")
rng = random.Random(SEED)
sample = rng.sample(train_rows, N_SAMPLE)

# Use the same RUBRIC and TMPL from your MLP_Joint_Judge Cell 1
# (copy them here before running, or import from that notebook)

client = genai.Client(vertexai=True, project=GCLOUD_PROJECT, location=GCLOUD_LOCATION)

def call_j(prompt):
    try:
        r = client.models.generate_content(model=JUDGE_MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=4000))
        raw = r.text if hasattr(r, "text") and r.text else None
        if not raw: return None
        s = raw.find("{")
        if s < 0: return None
        d = 0
        for i in range(s, len(raw)):
            if raw[i] == "{": d += 1
            elif raw[i] == "}": d -= 1
            if d == 0:
                try: return json.loads(raw[s:i+1])
                except: return None
        return None
    except Exception as e:
        if "429" in repr(e): time.sleep(60)
        return None

done = set()
if os.path.exists(OUT_FILE):
    for line in open(OUT_FILE):
        try:
            o = json.loads(line)
            if o.get("status") == "ok": done.add(o["id"])
        except: pass

fout = open(OUT_FILE, "a", encoding="utf-8")
for i, row in enumerate(sample):
    sid = row["id"]
    if sid in done: continue
    answer = row["teacher_text"]
    anon = {"A1": answer}
    tmpl = '"A1": {"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}'
    answers_block = f"--- A1 ---\n{answer}\n"
    prompt = TMPL.format(
        question=id2prompt[sid], n_answers=1, labels="A1",
        answers_block=answers_block, rubric=RUBRIC.format(n_answers=1),
        json_template=tmpl,
    )
    parsed = call_j(prompt)
    rec = {"id": sid, "status": "ok" if parsed and "A1" in parsed else "fail",
           "scores": {"teacher_TRAIN": parsed["A1"]} if parsed and "A1" in parsed else {}}
    fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
    fout.flush()
    if (i+1) % 10 == 0: print(f"  {i+1}/{N_SAMPLE}")
fout.close()

# Load + compute means
METRICS = ["decision_accuracy","safety_score","clinical_correctness","completeness","coherence","format_compliance"]
scores = {m: [] for m in METRICS}
for line in open(OUT_FILE):
    o = json.loads(line)
    if o.get("status") == "ok":
        for m in METRICS:
            if m in o["scores"].get("teacher_TRAIN", {}):
                scores[m].append(float(o["scores"]["teacher_TRAIN"][m]))

print(f"\n=== TRAINING teacher (N={len(scores['coherence'])}) ===")
for m in METRICS:
    if scores[m]:
        print(f"  {m:25s}: {sum(scores[m])/len(scores[m]):.3f}")

Loaded 6235 valid training teacher rows
  10/50
  20/50
  30/50
  40/50
  50/50

=== TRAINING teacher (N=50) ===
  decision_accuracy        : 4.600
  safety_score             : 4.760
  clinical_correctness     : 4.480
  completeness             : 4.620
  coherence                : 4.980
  format_compliance        : 5.000
